In [ ]:
# @title

import pandas as pd
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from matplotlib import lines
from datetime import datetime

dataset = pd.read_csv("https://raw.githubusercontent.com/jordanlewis9/clean-auditor-data/refs/heads/main/shortened-data.csv", index_col=False)

In [ ]:
# @title
wall_counts = dataset['WALL'].value_counts()
rare_walls = wall_counts[wall_counts < 100].index

height_counts = dataset['HEIGHT'].value_counts()
rare_heights = height_counts[height_counts < 100].index

dataset['WALL'] = dataset['WALL'].replace(rare_walls, 'OTHER')
dataset['WALL'] = dataset['WALL'].fillna('OTHER')
dataset['HEIGHT'] = dataset['HEIGHT'].replace(rare_heights, 'OTHER')
dataset['HEIGHT'] = dataset['HEIGHT'].fillna('OTHER')

dataset_visuals = dataset.copy()

grade_mapping = {
    'E-1': 1,
    'E': 2,
    'E+1': 3,
    'E+2': 4,
    'D-2': 5,
    'D-1': 6,
    'D': 7,
    'D+1': 8,
    'D+2': 9,
    'C-2': 10,
    'C-1': 11,
    'C': 12,
    'C+1': 13,
    'C+2': 14,
    'B-1': 15,
    'B': 16,
    'B+1': 17,
    'B+2': 18,
    'A-1': 19,
    'A': 20,
    'A+1': 21,
    'A+2': 22,
    'AA-': 23,
    'AA': 24,
    'AA+': 25,
    'X-': 26,
    'X': 27,
    'X+': 28
}

dataset['GRADE'] = dataset['GRADE'].map(grade_mapping)

dataset = pd.get_dummies(dataset, columns=['WALL'], dtype=int)
dataset = pd.get_dummies(dataset, columns=['ZIPCODE'], dtype=int)
dataset = pd.get_dummies(dataset, columns=['SCHOOL'], dtype=int)
dataset = pd.get_dummies(dataset, columns=['HEIGHT'], dtype=int)
dataset['NOSTORY'] = dataset['NOSTORY'] / 10

x = dataset.drop(columns=['APPRBLD'])
y = dataset['APPRBLD']

In [ ]:
# @title
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=67)
regressor = RandomForestRegressor(n_estimators = 50, max_depth=20, random_state = 0)
regressor.fit(x_train, y_train)
model_columns = x_train.columns

In [ ]:
# @title
y_pred = regressor.predict(x_test)
test_mae = mean_absolute_error(y_test, y_pred)
test_r2 = r2_score(y_test, y_pred)
train_y_pred = regressor.predict(x_train)
train_mae = mean_absolute_error(y_train, train_y_pred)
train_r2 = r2_score(y_train, train_y_pred)

In [ ]:
# @title
importances = regressor.feature_importances_
feature_names = x_train.columns
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances * 100
})

combined_importance_df = pd.DataFrame(columns=['Feature', 'Importance'])
zipcode_total = 0
school_total = 0
height_total = 0
wall_total = 0

for index, row in importance_df.iterrows():
    if 'ZIPCODE_' in row['Feature']:
        zipcode_total += row['Importance']
    elif 'SCHOOL_' in row['Feature']:
        school_total += row['Importance']
    elif 'HEIGHT_' in row['Feature']:
        height_total += row['Importance']
    elif 'WALL_' in row['Feature']:
        wall_total += row['Importance']
    else:
        combined_importance_df.loc[len(combined_importance_df)] = [row['Feature'], row['Importance']]

combined_importance_df.loc[len(combined_importance_df)] = ['ZIPCODE', zipcode_total]
combined_importance_df.loc[len(combined_importance_df)] = ['SCHOOL', school_total]
combined_importance_df.loc[len(combined_importance_df)] = ['HEIGHT', height_total]
combined_importance_df.loc[len(combined_importance_df)] = ['WALL', wall_total]
combined_importance_df = combined_importance_df.sort_values(by='Importance', ascending=False)

In [ ]:
# @title
scatter_checkbox = widgets.Checkbox(
    description="Toggle Area/Appraisal Scatterplot",
    value=False,
    style={'description_width': 'initial'}
)

scatter_output = widgets.Output()

def toggle_scatter(e):
    if scatter_checkbox.value == False:
        with scatter_output:
            scatter_output.clear_output()
            return
    else:
        fig, ax = plt.subplots(figsize=(14, 9))
        ax.scatter(dataset_visuals['AREA_A'], dataset_visuals['APPRBLD'], label="Appraisal Value")
        m, b = np.polyfit(dataset_visuals['AREA_A'], dataset_visuals['APPRBLD'], 1)
        ax.plot(dataset_visuals['AREA_A'], m*dataset_visuals['AREA_A'] + b, color='red', linestyle='-', linewidth=2, label='Regression Line')
        ax.yaxis.grid(True, linestyle='--', which='major', color='grey', alpha=.25)
        plt.xlabel('Area (in sqft)')
        plt.ylabel('Appraisal Value')
        plt.title('Area vs Appraisal Value')
        ax.yaxis.set_major_formatter('${x:1.0f}')
        ax.legend(loc='best', fontsize=10)

        with scatter_output:
            scatter_output.clear_output()
            plt.show()

scatter_checkbox.observe(toggle_scatter, names='value')

In [ ]:
# @title
unique_zips = dataset_visuals['ZIPCODE'].unique().tolist()
unique_zips.sort()
zips_medians = dataset_visuals.groupby("ZIPCODE")['APPRBLD'].median()
dropdown_one = widgets.Dropdown(
    options=unique_zips,
    value=unique_zips[0],
    description='Zipcode 1: '
)
dropdown_two = widgets.Dropdown(
    options=unique_zips,
    value=unique_zips[0],
    description='Zipcode 2: '
)
zipcode_button = widgets.Button(
    description="Compare Zipcodes"
)
error_html = widgets.HTML(
    value="<div style='color: red; font-size: 20px; font-weight: 700;'>Zipcodes cannot be the same!</div>"
)
output = widgets.Output()

def compare_zipcodes(e):
    zip_one = dropdown_one.value
    zip_two = dropdown_two.value

    if zip_one == zip_two:
        with output:
            output.clear_output()
            display(error_html)
        return

    zip_one_items = dataset_visuals[dataset_visuals['ZIPCODE'] == zip_one]['APPRBLD']
    zip_two_items = dataset_visuals[dataset_visuals['ZIPCODE'] == zip_two]['APPRBLD']
    zip_one_values = zip_one_items.values
    zip_two_values = zip_two_items.values
    zip_one_median = zip_one_items.median()
    zip_two_median = zip_two_items.median()

    with output:
        output.clear_output()
        box_data = [zip_one_values, zip_two_values]
        fig, ax = plt.subplots(figsize=(8, 9))
        box_plot = ax.boxplot(box_data, tick_labels=[str(zip_one), str(zip_two)], patch_artist=True, medianprops={'color': 'red', 'linewidth': 2})

        for patch in box_plot['boxes']:
            patch.set_facecolor('lightblue')

        ax.yaxis.set_major_formatter('${x:1.0f}')
        ax.yaxis.grid(True, linestyle='--', which='major', color='grey', alpha=.25)
        ax.set_title(f"Appraisal Value Comparison Between {zip_one} and {zip_two}")
        plt.text(x=1.1, y=zip_one_median, s=f'${zip_one_median:,.0f}', color='black', verticalalignment='center', fontweight='bold')
        plt.text(x=2.1, y=zip_two_median, s=f'${zip_two_median:,.0f}', color='black', verticalalignment='center', fontweight='bold')
        plt.xlabel('Compared Zipcodes')
        plt.ylabel('Appraisal Value')
        plt.show()

dropdown_box = widgets.HBox([dropdown_one, dropdown_two, zipcode_button])
zipcode_button.on_click(compare_zipcodes)

In [ ]:
# @title
unique_schools = dataset_visuals['SCHOOL'].unique().tolist()
unique_schools.sort()
schools_medians = dataset_visuals.groupby("SCHOOL")['APPRBLD'].median()

unique_grades = dataset_visuals['GRADE'].unique().tolist()
total_grades = len(unique_grades)
sorted_grades = [None] * total_grades
for grade in unique_grades:
    if grade in grade_mapping:
        sorted_grades[grade_mapping[grade] - 1] = grade

grades_medians = dataset_visuals.groupby("GRADE")['APPRBLD'].median()
all_dropdown = widgets.Dropdown(
    options=["Schools", "Zipcodes", "Grades"],
    value="Schools",
    description="Choose Feature: ",
    style={'description_width': 'initial'}
)
all_button = widgets.Button(
    description="View Median Appraisals",
    layout=widgets.Layout(width='200px', height='28px')
)
all_output = widgets.Output()

def show_appraisals(e):
    appraisals_x = []
    appraisals_y = []
    if all_dropdown.value == "Schools":
        for school_code, school_appraisal in schools_medians.items():
            appraisals_x.append(str(school_code))
            appraisals_y.append(school_appraisal)
    elif all_dropdown.value == "Grades":
        appraisals_x = [None] * len(grades_medians)
        appraisals_y = [None] * len(grades_medians)
        for grade, grade_appraisal in grades_medians.items():
            appraisals_x[grade_mapping[grade] - 1] = grade
            appraisals_y[grade_mapping[grade] - 1] = grade_appraisal
    else:
        for zipcode, zipcode_appraisal in zips_medians.items():
            appraisals_x.append(str(zipcode))
            appraisals_y.append(zipcode_appraisal)
    with all_output:
        all_output.clear_output()
        fig, ax = plt.subplots(figsize=(14, 7))
        bars = ax.bar(appraisals_x, appraisals_y, color=['red', 'blue', 'green'], width=1)
        ax.set_title(f"Median Building Appraisal Values for all {all_dropdown.value} in Franklin County")
        if all_dropdown.value == "Schools":
            ax.bar_label(bars, fmt="${:,.0f}")
            ax.set_xlabel("Schools")
        elif all_dropdown.value == "Grades":
            ax.bar_label(bars, fmt="${:,.0f}", rotation=90)
            ax.set_xlabel("Grades")
        else:
            ax.set_xlabel("Zipcodes")
            ax.bar_label(bars, fmt="${:,.0f}", rotation=90)
            plt.xticks(rotation=90)
        ax.set_ylabel("Appraisal Value")
        ax.yaxis.set_major_formatter('${x:1.0f}')
        plt.show()

all_box = widgets.HBox([all_dropdown, all_button])
all_button.on_click(show_appraisals)


In [ ]:
# @title
opening_html = widgets.HTML(
    value="<h1>Data Dashboard</h1>"
)

scatter_opening_html = widgets.HTML(
    value="<h2>Area vs Appraisal Value</h2>"
)

scatter_space = widgets.HTML(
    value="<div style='height: 30px;'></div>"
)

zipcode_html = widgets.HTML(
    value="<h2>Side-by-Side Zipcode Comparison</h2>"
)

zipcode_space = widgets.HTML(
    value="<div style='height: 30px;'></div>"
)

median_html = widgets.HTML(
    value="<h2>Variety Median Appraisal Values</h2>"
)

median_space = widgets.HTML(
    value="<div style='height: 30px;'></div>"
)

display(opening_html)
scatter_box = widgets.VBox([scatter_opening_html, scatter_checkbox, scatter_space, scatter_output])
zipcode_box = widgets.VBox([zipcode_html, dropdown_box, zipcode_space, output])
median_box = widgets.VBox([median_html, all_box, median_space, all_output])
display(scatter_box)
display(zipcode_box)
display(median_box)

HTML(value='<h1>Data Dashboard</h1>')

In [ ]:
# @title
prediction_html = widgets.HTML(
    value="<h1>Building Appraisal Predictor</h1>"
)

prediction_space = widgets.HTML(
    value="<div style='height: 30px;'></div>"
)

unique_heights = dataset_visuals['HEIGHT'].unique().tolist()
unique_heights.sort()

unique_walls = dataset_visuals['WALL'].unique().tolist()
unique_walls.sort()

area_input = widgets.BoundedIntText(
    value=2000,
    min=0,
    max=200000,
    step=1,
    description='Area in sqft: ',
    style={'description_width': 'initial'}
)

rooms_input = widgets.BoundedIntText(
    value=9,
    min=0,
    max=300,
    step=1,
    description='Number of Rooms: ',
    style={'description_width': 'initial'}
)

bedrooms_input = widgets.BoundedIntText(
    value=3,
    min=0,
    max=300,
    step=1,
    description='Number of Bedrooms: ',
    style={'description_width': 'initial'}
)

baths_input = widgets.BoundedIntText(
    value=2,
    min=0,
    max=30,
    step=1,
    description='Number of Bathrooms (Full Only): ',
    style={'description_width': 'initial' }
)

half_bath_input = widgets.BoundedIntText(
    value=0,
    min=0,
    max=30,
    step=1,
    description='Number of Half Baths: ',
    style={'description_width': 'initial'}
)

story_input = widgets.Dropdown(
    options=[1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0],
    value=1.0,
    description="Number of Stories: ",
    style={'description_width': 'initial'}
)

grade_input = widgets.Dropdown(
    options=sorted_grades,
    value=sorted_grades[0],
    description="Grade of Building: ",
    style={'description_width': 'initial'}
)

year_built_input = widgets.BoundedIntText(
    value=1990,
    min=1800,
    max=datetime.now().year,
    step=1,
    description='Year Built: ',
    style={'description_width': 'initial'}
)

zipcode_input = widgets.Dropdown(
    options=unique_zips,
    value=unique_zips[0],
    description='Zipcode: ',
    style={'description_width': 'initial'}
)

school_input = widgets.Dropdown(
    options=unique_schools,
    value=unique_schools[0],
    description='School Code: ',
    style={'description_width': 'initial'}
)

height_input = widgets.Dropdown(
    options=unique_heights,
    value=unique_heights[0],
    description='Type of Dwelling: ',
    style={'description_width': 'initial'}
)

wall_input = widgets.Dropdown(
    options=unique_walls,
    value=unique_walls[0],
    description='Building Material: ',
    style={'description_width': 'initial'}
)

ac_input = widgets.Checkbox(
    value=True,
    description='Has Air Conditioning?',
    style={'description_width': 'initial'}
)

predict_button = widgets.Button(
    description="Predict Appraisal Value",
    layout=widgets.Layout(width='200px', height='28px')
)

predict_output = widgets.Output()

def predict_appraisal(e):
    if area_input.value <= 0:
        with predict_output:
            predict_output.clear_output()
            print("Area cannot be negative")
        return
    if rooms_input.value < 0:
        with predict_output:
            predict_output.clear_output()
            print("Number of rooms cannot be negative")
        return
    if year_built_input.value < 1800 or year_built_input.value > datetime.now().year:
        with predict_output:
            predict_output.clear_output()
            print("Year built must be between 1800 and the current year")
        return
    predictor_metrics = {
        'AREA_A': area_input.value,
        'ROOMS': rooms_input.value,
        'BATHS': baths_input.value,
        'HBATHS': half_bath_input.value,
        'BEDRMS': bedrooms_input.value,
        'NOSTORY': story_input.value,
        'GRADE': grade_mapping[grade_input.value],
        'YEARBLT': year_built_input.value,
        'ZIPCODE_' + str(zipcode_input.value): 1,
        'SCHOOL_' + str(school_input.value): 1,
        'HEIGHT_' + str(height_input.value): 1,
        'WALL_' + str(wall_input.value): 1
    }

    predicted_df = pd.DataFrame(0, index=[0], columns=model_columns)
    for key, value in predictor_metrics.items():
        if key in predicted_df.columns:
            predicted_df.at[0, key] = value

    predicted_value = regressor.predict(predicted_df)[0]
    if predicted_value:
        with predict_output:
            predict_output.clear_output()
            print(f"Predicted Building Appraisal Value: ${predicted_value:,.0f}")

predict_row_1 = widgets.HBox([area_input, rooms_input, bedrooms_input])
predict_row_2 = widgets.HBox([baths_input, half_bath_input])
predict_row_3 = widgets.HBox([story_input, grade_input, year_built_input])
predict_row_4 = widgets.HBox([zipcode_input, school_input])
predict_row_5 = widgets.HBox([ac_input, height_input, wall_input])
predict_row_6 = widgets.HBox([predict_button])
predict_box = widgets.VBox([predict_row_1, predict_row_2, predict_row_3, predict_row_4, predict_row_5, predict_row_6])
predict_row_1.layout.width = '100%'
predict_row_2.layout.width = '100%'
predict_row_3.layout.width = '100%'
predict_row_4.layout.width = '100%'
predict_row_5.layout.width = '100%'
predict_row_6.layout.width = '100%'
predict_row_1.layout.justify_content = 'space-between'
predict_row_2.layout.justify_content = 'space-between'
predict_row_3.layout.justify_content = 'space-between'
predict_row_4.layout.justify_content = 'space-between'
predict_row_5.layout.justify_content = 'space-between'
predict_row_6.layout.justify_content = 'space-between'
predict_row_1.layout.margin = '0px 0px 20px 0px'
predict_row_2.layout.margin = '0px 0px 20px 0px'
predict_row_3.layout.margin = '0px 0px 20px 0px'
predict_row_4.layout.margin = '0px 0px 20px 0px'
predict_row_5.layout.margin = '0px 0px 20px 0px'
predict_row_6.layout.margin = '0px 0px 20px 0px'
display(prediction_html)
display(prediction_space)
display(predict_box)
predict_button.on_click(predict_appraisal)
display(predict_output)


HTML(value='<h1>Building Appraisal Predictor</h1>')

HTML(value="<div style='height: 30px;'></div>")

Output()

In [ ]:
# @title
model_columns = x_train.columns

miscellaneous_html = widgets.HTML(
    value="<h1>Miscellaneous Displays</h1>"
)

validation_html = widgets.HTML(
    value="<h2>Validation Metrics</h2>"
)

validation_space = widgets.HTML(
    value="<div style='height: 30px;'></div>"
)

validation_checkbox = widgets.Checkbox(
    description="Show Validation Metrics",
    value=False,
    style={'description_width': 'initial'}
)

validation_output = widgets.Output()

importances_html = widgets.HTML(
    value="<h2>Feature Importances</h2>"
)

importances_space = widgets.HTML(
    value="<div style='height: 30px;'></div>"
)

importances_checkbox = widgets.Checkbox(
    description="Show Feature Importances",
    value=False,
    style={'description_width': 'initial'}
)

importances_output = widgets.Output()

def toggle_validation(e):
    if validation_checkbox.value == False:
        with validation_output:
            validation_output.clear_output()
            return
    else:
        with validation_output:
            validation_output.clear_output()
            print(f"Test MAE: {test_mae}")
            print(f"Test R2: {test_r2}")
            print(f"Train MAE: {train_mae}")
            print(f"Train R2: {train_r2}")
            average_appraisal = y.mean()
            print(f"Average Value: {average_appraisal}")

def toggle_importances(e):
    if importances_checkbox.value == False:
        with importances_output:
            importances_output.clear_output()
            return
    else:
        with importances_output:
            importances_output.clear_output()
            fig, ax = plt.subplots(figsize=(12, 6))
            ax.pie(combined_importance_df['Importance'], labels=combined_importance_df['Feature'], autopct='%1.1f%%', startangle=90)
            ax.set_title('Feature Importance')
            plt.show()

            print(combined_importance_df)

validation_checkbox.observe(toggle_validation, names='value')
importances_checkbox.observe(toggle_importances, names='value')

display(miscellaneous_html)
display(validation_html)
display(validation_space)
display(validation_checkbox)
display(validation_output)
display(importances_html)
display(importances_space)
display(importances_checkbox)
display(importances_output)

HTML(value='<h1>Miscellaneous Displays</h1>')

HTML(value='<h2>Validation Metrics</h2>')

HTML(value="<div style='height: 30px;'></div>")

Checkbox(value=False, description='Show Validation Metrics', style=DescriptionStyle(description_width='initial…

Output()

HTML(value='<h2>Feature Importances</h2>')

HTML(value="<div style='height: 30px;'></div>")

Checkbox(value=False, description='Show Feature Importances', style=DescriptionStyle(description_width='initia…

Output()